# Monte Carlo Methods: Modular Generalized Policy Iteration (GPI)

This notebook provides a pedagogical demonstration of Monte Carlo (MC) methods using **Generalized Policy Iteration (GPI)**. We have broken the algorithm into modular components to clearly illustrate the inputs and outputs of each phase.

## 1. Mathematical Foundations

### The Return ($G_t$)
The goal of an RL agent is to maximize the expected return. The return is the total discounted reward from time step $t$:
$$G_t \doteq R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$
where $\gamma \in [0, 1]$ is the discount rate.

### Monte Carlo Prediction (Policy Evaluation)
In MC Prediction, we estimate the action-value function $q_\pi(s, a)$ by averaging the returns observed after visiting state $s$ and taking action $a$ **across many independent episodes**:
$$Q(s, a) \approx \mathbb{E}_\pi [G_t | S_t = s, A_t = a]$$

**Key Characteristic**: Unlike Dynamic Programming or TD learning, MC methods **wait until the end of an episode** to calculate the return $G_t$ and perform an update. The "average" is the empirical mean of these returns over time.

### Monte Carlo Control (Policy Improvement)
Once we have an estimate of $Q(s, a)$, we improve the policy by making it greedy with respect to the current value function:
$$\pi(s) \doteq \arg\max_a Q(s, a)$$

### Generalized Policy Iteration (GPI)
GPI consists of two interacting processes: **Evaluation** (making $Q$ consistent with $\pi$) and **Improvement** (making $\pi$ greedy w.r.t. $Q$).

## 2. Environment Setup

We use a simple 3x3 grid to visualize the learning process clearly.
- **States**: 0 to 8 (Grid layout: 0-1-2, 3-4-5, 6-7-8)
- **Goal**: State 8 (Terminal state, Reward = +10)
- **Other Steps**: Reward = -1 (Encourages finding the shortest path)
- **Actions**: 0: Up, 1: Down, 2: Left, 3: Right

<svg width="400" height="400" viewBox="0 0 400 400" xmlns="http://www.w3.org/2000/svg">
  <style>
    .cell { fill: #f8f9fa; stroke: #343a40; stroke-width: 2; }
    .text { font-family: sans-serif; font-size: 16px; fill: #343a40; text-anchor: middle; }
    .reward { font-size: 12px; fill: #6c757d; font-style: italic; }
    .goal { fill: #d4edda; }
    .goal-text { fill: #155724; font-weight: bold; }
    .arrow { fill: none; stroke: #007bff; stroke-width: 1.5; marker-end: url(#arrowhead); opacity: 0.6; }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="0" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#007bff" />
    </marker>
  </defs>
  <rect x="50" y="50" width="100" height="100" class="cell" />
  <text x="100" y="80" class="text">State 0</text>
  <text x="100" y="100" class="text reward">R: -1</text>
  <rect x="150" y="50" width="100" height="100" class="cell" />
  <text x="200" y="80" class="text">State 1</text>
  <text x="200" y="100" class="text reward">R: -1</text>
  <rect x="250" y="50" width="100" height="100" class="cell" />
  <text x="300" y="80" class="text">State 2</text>
  <text x="300" y="100" class="text reward">R: -1</text>
  <rect x="50" y="150" width="100" height="100" class="cell" />
  <text x="100" y="180" class="text">State 3</text>
  <text x="100" y="200" class="text reward">R: -1</text>
  <rect x="150" y="150" width="100" height="100" class="cell" />
  <text x="200" y="180" class="text">State 4</text>
  <text x="200" y="200" class="text reward">R: -1</text>
  <rect x="250" y="150" width="100" height="100" class="cell" />
  <text x="300" y="180" class="text">State 5</text>
  <text x="300" y="200" class="text reward">R: -1</text>
  <rect x="50" y="250" width="100" height="100" class="cell" />
  <text x="100" y="280" class="text">State 6</text>
  <text x="100" y="300" class="text reward">R: -1</text>
  <rect x="150" y="250" width="100" height="100" class="cell" />
  <text x="200" y="280" class="text">State 7</text>
  <text x="200" y="300" class="text reward">R: -1</text>
  <rect x="250" y="250" width="100" height="100" class="cell goal" />
  <text x="300" y="280" class="text goal-text">State 8</text>
  <text x="300" y="300" class="text reward goal-text">R: +10</text>
  <path d="M 200 160 L 200 135" class="arrow" />
  <path d="M 200 220 L 200 245" class="arrow" />
  <path d="M 170 190 L 145 190" class="arrow" />
  <path d="M 230 190 L 255 190" class="arrow" />
</svg>

In [ ]:
import numpy as np
import random
from collections import defaultdict

class SimpleGridWorld:
    def __init__(self):
        self.size = 3
        self.goal = 8
    
    def reset(self, start_state):
        self.state = start_state
        return self.state
    
    def step(self, action):
        row, col = divmod(self.state, self.size)
        if action == 0: row = max(0, row - 1)    # Up
        elif action == 1: row = min(self.size - 1, row + 1) # Down
        elif action == 2: col = max(0, col - 1)  # Left
        elif action == 3: col = min(self.size - 1, col + 1) # Right
        
        self.state = row * self.size + col
        reward = 10 if self.state == self.goal else -1
        done = (self.state == self.goal)
        return self.state, reward, done

## 3. The "Initial Guess"

How do we start if we don't have any data? We start with an **arbitrary policy**. 
- In our code, we use `policy = defaultdict(lambda: 3)`, which means the agent initially thinks "Going Right" is the best move for every state.
- The diversity in our 1000s of episodes doesn't come from having 1000 different policies; it comes from **Exploring Starts** (starting the agent in every possible corner of the grid). 
- As soon as the first episode ends, the policy begins to change and become smarter.

## 4. Phase 1: Episode Generation (Experience)

**Why do we need a policy here?**
A common misconception is that MC just 'tries things randomly.' In reality, we need a **policy** to act as a guide. After the first 'Exploring Start' action, the agent follows its current best plan (the policy) to ensure it actually reaches the goal. Without a policy, the agent might wander forever and never generate a Return ($G$) for us to learn from.

**Input**: Environment, Current Policy, Start State, Start Action.
**Output**: A list of `(state, action, reward)` tuples representing the trajectory.

In [ ]:
def generate_episode(env, policy, start_state, start_action):
    episode = []
    state = env.reset(start_state)
    
    # Take the 'Exploring Start' action
    next_state, reward, done = env.step(start_action)
    episode.append((state, start_action, reward))
    state = next_state
    
    # Follow policy until terminal state
    while not done:
        action = policy[state]
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        
    return episode

## 5. Phase 2: Update Phase (Policy Evaluation)

**Input**: The generated Episode, $Q$-table, and Tracking structures.
**Output**: Updated $Q$-table (Modified in-place).

This method processes the episode **backwards** to calculate the return $G$ for each step and updates the empirical average in the $Q$-table.

In [ ]:
def update_q_values(episode, Q, returns_sum, returns_count, gamma=0.9):
    G = 0
    visited_sa = set()
    
    # Process episode backwards
    for t in range(len(episode) - 1, -1, -1):
        s, a, r = episode[t]
        G = gamma * G + r
        
        # First-visit MC check
        if (s, a) not in visited_sa:
            visited_sa.add((s, a))
            returns_sum[s][a] += G
            returns_count[s][a] += 1
            Q[s][a] = returns_sum[s][a] / returns_count[s][a]
    # Note: Q is modified in-place

## 6. Phase 3: Improvement Phase (Policy Improvement)

**Input**: Current $Q$-table.
**Output**: Updated Policy (Modified in-place).

This method makes the policy **greedy** with respect to the current $Q$-values.

In [ ]:
def improve_policy(Q, policy):
    for s in range(8): # For all non-terminal states
        policy[s] = np.argmax(Q[s])
    # Note: policy is modified in-place

## 7. Stitching it Together: The MC ES Algorithm

Now we see how the individual phases combine to form the complete Generalized Policy Iteration cycle.

In [ ]:
def run_mc_es_demonstration(num_iterations=10):
    env = SimpleGridWorld()
    
    # Initialize Memory
    Q = defaultdict(lambda: np.zeros(4))
    returns_sum = defaultdict(lambda: np.zeros(4))
    returns_count = defaultdict(lambda: np.zeros(4))
    policy = defaultdict(lambda: 3) # Initial: always move Right

    for i in range(num_iterations):
        # Step 1: Pick random start (Exploring Starts)
        s0 = random.choice(range(8))
        a0 = random.choice(range(4))
        
        # Phase 1: EVALUATION (Experience)
        episode = generate_episode(env, policy, s0, a0)
        
        # Phase 2: EVALUATION (Update)
        update_q_values(episode, Q, returns_sum, returns_count)
        
        # Phase 3: IMPROVEMENT
        improve_policy(Q, policy)
        
        print(f"Iteration {i+1}: Policy updated. Current Policy Grid:")
        print(np.array([policy[s] for s in range(9)]).reshape(3,3))
        print("-" * 30)

run_mc_es_demonstration(15)